# Chapter 9 – Entangling Intelligence

This notebook introduces the foundations of **quantum machine learning** through a series of hands-on examples built with the **NHANES** patient dataset.

You'll begin by encoding classical patient records into quantum states using **angle encoding** and visualizing the resulting feature maps on the **Bloch sphere**. Next, you'll compare patients using both **classical cosine similarity** and a **quantum kernel**, generating similarity matrices and heatmaps that highlight the differences between the two approaches.

The notebook then builds a complete **QAOA-based clustering workflow**. Patient similarities are transformed into a graph-partitioning problem, optimized using QAOA, and visualized as both a similarity map and a patient relationship graph.

By the end of the notebook, you'll have implemented a complete **hybrid classical-quantum machine learning workflow** while gaining practical experience with quantum feature maps, quantum kernels, quantum optimization, and the visualizations that help make these ideas intuitive.

### Code 9-1: Encoding Patient Features into a Quantum Circuit

In Chapter 6, we used NHANES-style patient data to build a large search space. Here we use the same dataset for a different purpose: learning.

A classical machine learning model can accept a row of patient data as a simple array. A quantum circuit needs that information translated into operations on qubits. Code 9-1 shows one of the simplest approaches, called **angle encoding**. Each feature is scaled into an angle, and that angle controls a rotation gate.

This gives us a concrete way to see the first challenge of quantum machine learning. Before the model can find patterns, the data has to enter the quantum system.

**Note:** This example requires Qiskit and Aer. Execute the package installation cell immediately below this markdown cell before running the code.

In [ ]:
!pip -q install qiskit qiskit-aer pylatexenc

In [ ]:
# === Code 9-1: Encoding Patient Features into a Quantum Circuit ===

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from qiskit import QuantumCircuit

NHANES_URL = "https://raw.githubusercontent.com/JerryCuomo/ThinkQuantum/main/datasets/nhanes_2026.csv"

# Step 1: Select a patient and the health features we want to learn from.
url = NHANES_URL
cols = ["ID", "Age", "BMI", "BPSysAve", "BPDiaAve", "TotChol", "Diabetes"]

df = pd.read_csv(url)[cols].dropna().reset_index(drop=True)
features = ["Age", "BMI", "BPSysAve", "BPDiaAve", "TotChol"]

gold_patients = df[(df.Age >= 50) & (df.BMI >= 30) & (df.BPSysAve >= 130)]
patient = gold_patients.sample(1, random_state=7)

# Step 2: Scale each feature into a rotation angle between 0 and π.
scaler = MinMaxScaler(feature_range=(0, np.pi))
angles = scaler.fit_transform(df[features]).take(patient.index, axis=0)[0]

encoding = pd.DataFrame({
    "Feature": features,
    "Value": patient[features].iloc[0].values,
    "Angle (radians)": np.round(angles, 4)
})

print("Selected patient:")
display(patient[["ID"] + features + ["Diabetes"]])

print("\nFeature encoding:")
display(encoding)

# Step 3: Build a quantum feature map using one qubit per feature.
qc = QuantumCircuit(len(features), name="PatientFeatureMap")

for qubit, angle in enumerate(angles):
    qc.ry(angle, qubit)

qc.draw("mpl")

#### Code 9-1B: Visualizing Angle Encoding

This example displays the encoded patient state on the Bloch sphere, providing a visual view of how classical feature values are transformed into quantum states.

In [ ]:
# === Code 9-1B: Patient Features After Angle Encoding ===

import numpy as np
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.set_axis_off()

# Bloch sphere
u = np.linspace(0, 2*np.pi, 40)
v = np.linspace(0, np.pi, 20)

x = np.outer(np.cos(u), np.sin(v))
y = np.outer(np.sin(u), np.sin(v))
z = np.outer(np.ones_like(u), np.cos(v))

ax.plot_wireframe(x, y, z, color="gray", alpha=0.15)

# Spread the states around the sphere only to make them easier to see.
# The actual RY-encoded states lie in the X-Z plane.
phis = np.linspace(0, 2*np.pi, len(features), endpoint=False)

for feature, theta, phi in zip(features, angles, phis):

    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    ax.scatter(x, y, z, s=280)

    ax.text(
        x * 1.25,
        y * 1.3,
        z * 1,
        feature,
        ha="center",
        va="center",
        fontsize=12
    )

ax.set_box_aspect([1, 1, 1])
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_zlim([-1.2, 1.2])

ax.set_title("Patient Features After Angle Encoding", fontsize=16, pad=20)

plt.tight_layout()
plt.show()

### Code 9-2: Comparing Patients with a Quantum Kernel

Code 9-2 compares three patient records using two different measures of similarity. Two patients come from the higher-risk population, while a third provides a healthier contrast.

The example first applies **cosine similarity** to the scaled feature vectors. It then encodes the same values as qubit rotations and computes a **quantum kernel** from the overlap of the resulting quantum states.

The final table places the scores side by side, making it easier to see how the classical and quantum representations evaluate the same patient relationships.


In [ ]:
# === Code 9-2: Comparing Patients with a Quantum Kernel ===

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Step 1: Select three patients to compare.
patient_a = patient  # Patient selected in Code 9-1.

# Select a second gold patient with a similar health profile.
patient_b = gold_patients.drop(patient_a.index).sample(1, random_state=11)

# Select a healthier patient for contrast.
comparison_pool = df[
    (df.Age < 40) &
    (df.BMI < 27) &
    (df.BPSysAve < 125)
]

patient_c = comparison_pool.sample(1, random_state=13)

patients = pd.concat([patient_a, patient_b, patient_c])
labels = ["Patient A", "Patient B", "Patient C"]

print("Selected patients:")
display(patients[["ID"] + features + ["Diabetes"]])

# Step 2: Scale the patient features using the encoder from Code 9-1.
X = patients[features]
X_scaled = scaler.transform(X)

# Step 3: Measure similarity in the classical feature space.
cosine_matrix = cosine_similarity(X_scaled)

cosine_df = pd.DataFrame(
    cosine_matrix,
    index=labels,
    columns=labels
)

print("\nClassical cosine similarity:")
display(cosine_df.round(4))

# Step 4: Build the same quantum feature map used in Code 9-1.

# Convert encoded feature values into a quantum state.
def build_feature_map(feature_angles):
    qc = QuantumCircuit(len(feature_angles))

    # Encode each feature as an RY rotation.
    for qubit, angle in enumerate(feature_angles):
        qc.ry(angle, qubit)

    return qc


# Compare two patient records using quantum-state overlap.
def quantum_kernel(feature_angles_a, feature_angles_b):

    # Convert both patient records into quantum states.
    state_a = Statevector.from_instruction(build_feature_map(feature_angles_a))
    state_b = Statevector.from_instruction(build_feature_map(feature_angles_b))

    # Compute the quantum-state overlap and return a similarity score.
    overlap = np.vdot(state_a.data, state_b.data)

    return np.abs(overlap) ** 2

# Step 5: Measure similarity between every pair of patients.
quantum_matrix = np.zeros((len(X_scaled), len(X_scaled)))

for i in range(len(X_scaled)):
    for j in range(len(X_scaled)):
        quantum_matrix[i, j] = quantum_kernel(X_scaled[i], X_scaled[j])

quantum_df = pd.DataFrame(
    quantum_matrix,
    index=labels,
    columns=labels
)

print("\nQuantum kernel similarity:")
display(quantum_df.round(4))

# Step 6: Compare the classical and quantum similarity scores.
pairs = [
    ("Patient A", "Patient B", 0, 1),
    ("Patient A", "Patient C", 0, 2),
    ("Patient B", "Patient C", 1, 2)
]

results = []

for left, right, i, j in pairs:
    results.append({
        "Pair": f"{left} vs {right}",
        "Cosine Similarity": cosine_matrix[i, j],
        "Quantum Kernel": quantum_matrix[i, j]
    })

comparison = pd.DataFrame(results)

print("\nSimilarity comparison:")
display(comparison.round(4))

#### Code 9-2B: Visualizing Similarity

This example displays the classical and quantum similarity matrices, making it easy to compare how each method measures relationships between the same patient records.

In [ ]:
# === Code 9-2B: Visualizing Patient Similarity ===

import matplotlib.pyplot as plt

feature_text = "Features: " + ", ".join(features)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.patch.set_facecolor("white")

plots = [
    (axes[0], cosine_matrix, "Classical Cosine Similarity"),
    (axes[1], quantum_matrix, "Quantum Kernel Similarity")
]

for ax, matrix, title in plots:
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title, fontsize=20, pad=24)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_yticklabels(labels, fontsize=13)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text_color = "white" if value > 0.65 else "black"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                    fontsize=16, fontweight="bold", color=text_color)

fig.suptitle("Comparing Patient Similarity", fontsize=24, y=0.96)
fig.text(0.5, 0.90, feature_text, ha="center", fontsize=14)

plt.subplots_adjust(wspace=0.30, top=0.83, bottom=0.12)
plt.show()

### Code 9-3: Clustering Patients with QAOA

Code 9-3 brings together the major ideas developed throughout this chapter. Starting with eight patient records, we measure similarity using the quantum kernel, transform those relationships into a Max-Cut optimization problem, and use QAOA to discover two natural patient groups.

This example also illustrates the hybrid nature of quantum machine learning. Classical code prepares the data, constructs the optimization problem, and refines the QAOA parameters, while the quantum circuit evaluates candidate solutions. The final bitstring is then translated back into patient clusters.

Four small helper functions at the top of the cell handle the QAOA mechanics used in Step 4, which keeps the six steps below easy to follow. Run this cell first. The next one draws the pictures.

In [ ]:
# === Code 9-3: Clustering Patients with QAOA (helpers and Steps 1 to 5) ===

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# --- Helper functions used by Step 4 ---

# Score a candidate partition: the total dissimilarity placed across the cut.
def cut_value(bitstring, edges):
    return sum(weight for i, j, weight in edges if bitstring[i] != bitstring[j])

# Build a single-layer QAOA circuit for the weighted Max-Cut problem.
def build_qaoa_circuit(n, edges, gamma, beta):
    qc = QuantumCircuit(n)
    qc.h(range(n))                    # Start with every partition in superposition.

    for i, j, weight in edges:        # Cost layer: favor cuts with larger weights.
        qc.cx(i, j)
        qc.rz(2 * gamma * weight, j)
        qc.cx(i, j)

    for qubit in range(n):            # Mixer layer: keep exploring alternatives.
        qc.rx(2 * beta, qubit)

    return qc

# Return the probability of every partition, with bitstrings in patient order.
def qaoa_probabilities(gamma, beta, n, edges):
    state = Statevector.from_instruction(build_qaoa_circuit(n, edges, gamma, beta))

    # Qiskit writes qubit 0 last, so reverse each bitstring to read P0 first.
    return {bits[::-1]: prob for bits, prob in state.probabilities_dict().items()}

# Average cut value across everything the circuit might produce.
def expected_cut(params, n, edges):
    probs = qaoa_probabilities(*params, n, edges)
    return sum(prob * cut_value(bits, edges) for bits, prob in probs.items())


# Step 1: Select eight patients for clustering.
high_risk = gold_patients.sample(4, random_state=7)

low_risk = df[
    (df.Age < 40) &        # Younger patients.
    (df.BMI < 27) &        # Lower BMI range.
    (df.BPSysAve < 125)    # Lower systolic blood pressure.
].sample(4, random_state=13)

patients = pd.concat([high_risk, low_risk]).reset_index(drop=True)
patient_labels = [f"P{i}" for i in range(len(patients))]
X_scaled = scaler.transform(patients[features])

print("Patients selected:")
display(patients[["ID"] + features + ["Diabetes"]])

# Step 2: Build a quantum-kernel similarity matrix.
n = len(patients)
similarity = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity[i, j] = quantum_kernel(X_scaled[i], X_scaled[j])

similarity_df = pd.DataFrame(similarity, index=patient_labels, columns=patient_labels)

print("\nQuantum-kernel similarity:")
display(similarity_df.round(3))

# Step 3: Convert similarity into a Max-Cut optimization problem.
dissimilarity = 1 - similarity
np.fill_diagonal(dissimilarity, 0)

edges = [
    (i, j, dissimilarity[i, j])
    for i in range(n)
    for j in range(i + 1, n)
]

# Step 4: Run QAOA by optimizing the circuit angles.
result = minimize(
    lambda params: -expected_cut(params, n, edges),
    x0=[0.8, 0.4],          # Starting guesses for gamma and beta.
    method="COBYLA",        # Derivative-free classical optimizer.
    options={"maxiter": 80} # Small iteration budget for Colab.
)

gamma, beta = result.x
probs = qaoa_probabilities(gamma, beta, n, edges)

best_bitstring = max(probs, key=probs.get)
best_score = cut_value(best_bitstring, edges)

print("\nBest QAOA bitstring:", best_bitstring)
print("Probability:        ", f"{probs[best_bitstring]:.1%}")
print("Best cut score:     ", round(best_score, 3))

# Step 5: Interpret the bitstring as two patient clusters.
patients["Cluster"] = ["Cluster A" if bit == "0" else "Cluster B" for bit in best_bitstring]

assignment = pd.DataFrame({
    "Patient": patient_labels,
    "Bit": list(best_bitstring),
    "Cluster": patients["Cluster"]
})

print("\nQAOA cluster assignment:")
display(assignment)

#### Code 9-3B: Visualizing the Clusters

Numbers only go so far, so this cell draws three views of the same eight patients: bar charts of the five health features, a graph of the strongest similarities colored by cluster, and a heatmap of the full similarity matrix. As you look at the bar charts, notice which measurements separate the two groups and which ones don't. QAOA had to find the groups without being told. Run Code 9-3 first, since these plots use its results.

In [ ]:
# === Code 9-3B: Visualizing the Clusters (plots and Step 6) ===

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

# Draw one small bar chart per feature, with the eight patients as bars.
def draw_patient_features(patients, features, feature_titles, patient_labels, group_sizes=(4, 4)):
    # Leave a gap between the two groups so they read as separate sets of bars.
    positions = np.r_[np.arange(group_sizes[0]), np.arange(group_sizes[1]) + group_sizes[0] + 0.8]

    # Hatching repeats the color cue, so the groups stay distinct in grayscale print.
    colors = ["#1f497d"] * group_sizes[0] + ["#b45f06"] * group_sizes[1]
    hatches = [""] * group_sizes[0] + ["///"] * group_sizes[1]

    fig, axes = plt.subplots(2, 3, figsize=(11, 6.6))
    axes = axes.ravel()

    for ax, feature, title in zip(axes, features, feature_titles):
        for pos, value, color, hatch in zip(positions, patients[feature], colors, hatches):
            ax.bar(pos, value, width=0.75, edgecolor=color, linewidth=1.2,
                   color=color if not hatch else "white", hatch=hatch)

        ax.set_title(title, fontsize=13, loc="left")
        ax.set_xticks(positions, labels=patient_labels, fontsize=10)
        ax.tick_params(axis="x", length=0)
        ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)

    # Use the sixth panel for the legend.
    axes[-1].axis("off")
    axes[-1].legend(
        handles=[
            Patch(facecolor=colors[0], edgecolor=colors[0], label="Higher-risk group (P0 to P3)"),
            Patch(facecolor="white", edgecolor=colors[-1], hatch="///", label="Younger group (P4 to P7)"),
        ],
        loc="center", frameon=False, fontsize=12
    )
    plt.tight_layout()
    plt.show()

# Draw patients as a graph: nodes are patients, edges are strong similarities.
def draw_patient_graph(similarity, clusters, patient_labels, threshold=0.55):
    count = len(patient_labels)

    graph = nx.Graph()
    graph.add_nodes_from(range(count))
    graph.add_edges_from(
        (i, j, {"weight": similarity[i, j]})
        for i in range(count)
        for j in range(i + 1, count)
        if similarity[i, j] >= threshold
    )

    node_colors = ["#1f497d" if c == "Cluster A" else "#b45f06" for c in clusters]
    edge_widths = [1 + 4 * graph[u][v]["weight"] for u, v in graph.edges]
    labels = dict(enumerate(patient_labels))
    pos = nx.spring_layout(graph, seed=7, weight="weight")

    plt.figure(figsize=(9, 9))
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=1100)
    nx.draw_networkx_edges(graph, pos, width=edge_widths, alpha=0.45)
    nx.draw_networkx_labels(
        graph, pos, labels=labels,
        font_weight="bold", font_color="white", font_size=16
    )
    plt.axis("off")
    plt.show()

# Draw the quantum-kernel similarity matrix as a heatmap.
def draw_similarity_heatmap(similarity, patient_labels):
    ticks = range(len(patient_labels))
    edges_of_cells = np.arange(-0.5, len(patient_labels), 1)

    fig, ax = plt.subplots(figsize=(5.3, 4.8))
    image = ax.imshow(similarity, cmap="Blues", vmin=0, vmax=0.9)

    ax.set_title("Quantum Similarity Map", fontsize=14, pad=10)
    ax.set_xticks(ticks, labels=patient_labels)
    ax.set_yticks(ticks, labels=patient_labels)

    # Light gridlines make the block structure easier to see.
    ax.set_xticks(edges_of_cells, minor=True)
    ax.set_yticks(edges_of_cells, minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)

    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Similarity")
    plt.tight_layout()
    plt.show()

# Step 6: Visualize the patients, the strongest similarities, and the clusters.
draw_patient_features(
    patients, features,
    ["Age (years)", "BMI (kg/m²)", "Systolic BP (mmHg)", "Diastolic BP (mmHg)", "Total Cholesterol (mmol/L)"],
    patient_labels
)
draw_patient_graph(similarity, patients["Cluster"], patient_labels)
draw_similarity_heatmap(similarity, patient_labels)